# Clean Storm Data from Observations, Add ERA5 Temperatures

For each SNOTEL and CDEC station in the study area, finds the GridMET storms at the station's grid cell and adds up the observed SWE gain during each one. Each storm then gets the ERA5 storm temperature of the basin-wide GridMET storm it overlaps, from the station's basin (or the nearest basin, for stations outside all basins).

Stations are kept if they fall inside the bounding box of the study basins, which is also the GridMET domain. Before adding up SWE gains, day-to-day jumps of 1000 mm or more are removed, the same rule used in the peak-SWE quality control.

## Inputs
| File | From | Used for |
|------|------|----------|
| `Data/Cleaned/SNOTEL/snotel_ds.nc4`, `CDEC/cdec_ds.nc4` | Observations notebook | Daily SWE [mm] and station locations |
| `Data/Cleaned/GridMET/storm_labels.nc4` | GridMET notebook | Storm IDs per grid cell and day |
| `Data/Cleaned/<basin>_storms_long.nc4` | Storm temperature notebook | Basin storms (full record) with ERA5 `Temperature` [K] |
| `config.py` / `utils.py` | | Basins and shapes / `remove_swe_jumps`, `swe_gains` |

## Outputs
`Data/Cleaned/SNOTEL/snotel_storm_ds.nc4` and `Data/Cleaned/CDEC/cdec_storm_ds.nc4`, one row per (station, storm):

| Variable | Description |
|----------|-------------|
| `Start_Date`, `Duration` | First storm day and number of storm days at the station's grid cell |
| `Delta_SWE` | Observed SWE gain over the storm [mm]; NaN if the station has no data during the storm |
| `Station`, `Latitude`, `Longitude`, `Elevation` | Station ID and location |
| `Basin` | Basin the station is in, or the nearest one |
| `Temperature` | ERA5 temperature of the overlapping basin storm [K]; NaN if none overlaps |

In [ ]:
# %% Imports and inputs
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

from config import data_folder, Basins, basin_names_list, shapes
from utils import remove_swe_jumps, swe_gains

cleaned = os.path.join(data_folder, 'Data/Cleaned')
basins = shapes[shapes['name'].isin(basin_names_list)]
minx, miny, maxx, maxy = basins.total_bounds  # also the GridMET domain

storm_labels = xr.open_dataset(os.path.join(cleaned, 'GridMET/storm_labels.nc4'))['StormLabel']
basin_storms = {b: xr.open_dataset(os.path.join(cleaned, f'{b}_storms_long.nc4')) for b in Basins}

In [ ]:
# %% Observations: stations in the study-area box; SWE gains after removing jumps
def load_obs(path):
    obs = xr.open_dataset(path)
    in_box = ((obs['Latitude'] >= miny) & (obs['Latitude'] <= maxy) &
              (obs['Longitude'] >= minx) & (obs['Longitude'] <= maxx))
    obs = obs.sel(Station_ID=in_box)
    obs['Delta_SWE'] = swe_gains(remove_swe_jumps(obs['SWE']))  # full daily record, before picking storm days
    return obs


snotel = load_obs(os.path.join(cleaned, 'SNOTEL/snotel_ds.nc4'))
cdec = load_obs(os.path.join(cleaned, 'CDEC/cdec_ds.nc4'))
print(f"{snotel.sizes['Station_ID']} SNOTEL stations, {cdec.sizes['Station_ID']} CDEC stations")

In [ ]:
# %% Per-station storms and ERA5 temperature
def assign_basins(lat, lon):
    """Basin each point is in, or the nearest basin if it's outside all of them."""
    name_to_basin = dict(zip(basin_names_list, Basins))
    pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(lon, lat), crs='EPSG:4326').to_crs(3310)
    nearest = gpd.sjoin_nearest(pts, basins[['name', 'geometry']].to_crs(3310), how='left')
    return nearest.groupby(level=0)['name'].first().map(name_to_basin).values  # .first(): ties


def station_storms(obs):
    """One row per (station, GridMET storm at the station's nearest grid cell)."""
    station_basin = dict(zip(obs.Station_ID.values,
                             assign_basins(obs['Latitude'].values, obs['Longitude'].values)))
    frames = []
    for sid in obs.Station_ID.values:
        st = obs.sel(Station_ID=sid)
        lat, lon = st['Latitude'].item(), st['Longitude'].item()
        labels = storm_labels.sel(Latitude=lat, Longitude=lon, method='nearest').dropna('Day')
        df = pd.DataFrame({
            'Label': labels.values,
            'Day': labels['Day'].values,
            'Delta_SWE': st['Delta_SWE'].reindex(Day=labels['Day']).values,
        })
        g = df.groupby('Label')
        storms = g.agg(Start_Date=('Day', 'min'), Duration=('Day', 'size'))
        storms['Delta_SWE'] = g['Delta_SWE'].sum(min_count=1)  # NaN if no observations during the storm
        frames.append(storms.assign(Station=sid, Longitude=lon, Latitude=lat,
                                    Elevation=st['Elevation'].item(), Basin=station_basin[sid]))
    return pd.concat(frames, ignore_index=True).rename_axis('Storm_Label')


def overlapping_basin_storm(starts, ends, basin_of_row, var='Temperature'):
    """`var` of the basin storm that overlaps each [start, end]; NaN if none does.
    Basin storms are separate runs of consecutive days, so at most one starts first in the overlap."""
    out = np.full(len(starts), np.nan)
    for basin, ds in basin_storms.items():
        rows = basin_of_row == basin
        if not rows.any():
            continue
        order = np.argsort(ds['Start_Date'].values)
        b_start = ds['Start_Date'].values[order]
        b_end = b_start + (ds['Duration'].values[order] - 1).astype('timedelta64[D]')
        values = ds[var].values[order]

        i = np.searchsorted(b_end, starts[rows])        # first basin storm ending on/after the start
        j = np.minimum(i, len(b_start) - 1)
        overlaps = (i < len(b_start)) & (b_start[j] <= ends[rows])
        out[rows] = np.where(overlaps, values[j], np.nan)
    return out


for obs, network in [(snotel, 'SNOTEL'), (cdec, 'CDEC')]:
    df = station_storms(obs)
    ends = (df['Start_Date'] + pd.to_timedelta(df['Duration'] - 1, unit='D')).values
    df['Temperature'] = overlapping_basin_storm(df['Start_Date'].values, ends, df['Basin'].values)

    storm_ds = xr.Dataset.from_dataframe(df)
    storm_ds.to_netcdf(os.path.join(cleaned, network, f'{network.lower()}_storm_ds.nc4'))
    print(f"{network}: {storm_ds.sizes['Storm_Label']} station-storms, "
          f"{int(storm_ds['Temperature'].isnull().sum())} without an overlapping basin storm")